In [1]:
import cadquery as cq
import numpy as np
# import matplotlib.pyplot as plt
from cadquery import Edge, Vector, Wire

from OCP.Geom import Geom_BSplineCurve
from OCP.TColgp import TColgp_Array1OfPnt
from OCP.TColStd import TColStd_Array1OfReal, TColStd_Array1OfInteger
from OCP.gp import gp_Pnt, gp_Vec,gp_Dir, gp_Ax2
from OCP.TopoDS import TopoDS_Edge
from OCP.BRep import BRep_Builder
from OCP.BRepTools import BRepTools
from OCP.BRepBuilderAPI import BRepBuilderAPI_MakeWire, BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeFace

In [ ]:
# Building curve with periodic boundary condition.
ctrl_pts_xyz = [
    # (  X,     Y,     Z  )
    ( 1.00,  0.50,  0.00),   # P00 — start, bottom-right
    ( 1.20,  0.42, -0.80),   # P01 — descending right
    ( 1.60,  0.28, -1.50),   # P02 — bottom trough
    ( 2.10,  0.08, -1.20),   # P03 — sweeping left-bottom
    ( 2.50,  0.00,  0.00),   # P04 — leftmost point
    ( 2.30, -0.22,  1.00),   # P05 — sweeping left-top
    ( 1.80, -0.45,  1.50),   # P06 — top crest
    ( 1.30, -0.50,  1.30),   # P07 — upper right descent
    ( 1.00, -0.35,  0.80),   # P08 — right side, upper
    ( 0.95, -0.10,  0.50),   # P09 — inside the notch concavity
    ( 1.00,  0.15,  0.20),   # P10 — exiting the notch
    ( 1.00,  0.50,  0.00),   # P11 — back to start (closure)
]

unique_pts = ctrl_pts_xyz[:-1] 
degree = 4
n= len(unique_pts)

poles = TColgp_Array1OfPnt(1, n)
for i, (x, y, z) in enumerate(unique_pts, start=1):
    poles.SetValue(i,gp_Pnt(x,y,z))

n_knots = n+1

knots = TColStd_Array1OfReal(1, n_knots)
for i in range(n_knots):
    knots.SetValue(i + 1, float(i))


mults = TColStd_Array1OfInteger(1, n_knots)
for i in range(1, n_knots + 1):
    mults.SetValue(i, 1) 

is_periodic = True

curve = Geom_BSplineCurve(poles, knots, mults, degree, is_periodic)

tmin = curve.FirstParameter()
tmax = curve.LastParameter()

print(f"tmin: {tmin}, tmax: {tmax}")

OCP runs
tmin: 0.0, tmax: 11.0


In [ ]:
# Helper functions to convert point and vector to numpy arrays

def to_np_pnt(p: gp_Pnt)  -> np.ndarray:
    return np.array([p.X(), p.Y(), p.Z()])
 
def to_np_vec(v: gp_Vec)  -> np.ndarray:
    return np.array([v.X(), v.Y(), v.Z()])
 
def normalize(v: np.ndarray) -> np.ndarray:
    m = np.linalg.norm(v)
    return v / m if m > 1e-14 else v


In [ ]:
# Check dot product of omega_hat with T0 and T1 and dot product of T0 and N0 to confirm orthogonality

dt = (tmax - tmin)/100

p0 = to_np_pnt(curve.Value(tmin))
T0 = normalize(to_np_vec(curve.DN(tmin, 1)))
N0 = normalize(to_np_vec(curve.DN(tmin, 2)))
T1 = normalize(to_np_vec(curve.DN(tmin+dt, 1)))

omega = np.cross(T0, T1)
omega_mag = np.linalg.norm(omega)
omega_hat = normalize(omega)

dot_omega_T0 = np.dot(omega_hat, T0)
dot_omega_T1 = np.dot(omega_hat, T1)
dot_T0_N0 = np.dot(T0, N0)

print(f"Dot product of omega_hat and T0: {dot_omega_T0:.6f}")
print(f"Dot product of omega_hat and T1: {dot_omega_T1:.6f}")
print(f"Dot product of T0 and N0: {dot_T0_N0:.6f}")

Dot product of omega_hat and T0: -0.000000
Dot product of omega_hat and T1: -0.000000
Dot product of T0 and N0: -0.602662


In [ ]:
# All the tangents on the curve

N = 100
s_samples = np.linspace(tmin, tmax, N, endpoint=True)

tangents = []
for t in s_samples:
    dpds = curve.DN(t, 1)
    T  = np.array([dpds.X(), dpds.Y(), dpds.Z()])
    T  = T / np.linalg.norm(T)
    tangents.append(T)

tangents = np.array(tangents)   # shape (200, 3)
print(f"Tangents shape: {tangents.shape}")
print(f"First T: {tangents[0]}")
print(f"Last  T: {tangents[-1]}")

Tangents shape: (100, 3)
First T: [ 0.5597184  -0.2044189  -0.80307423]
Last  T: [ 0.5597184  -0.2044189  -0.80307423]


In [ ]:
# Compute omegas for each pair of consecutive tangents
omegas = []
for i in range(len(tangents) - 1):
    T0    = tangents[i]
    T1    = tangents[i + 1]
    omega = np.cross(T0, T1)
    omega = omega / np.linalg.norm(omega)
    omegas.append(omega)

omegas = np.array(omegas)   # shape (199, 3)
print(f"Omegas shape: {omegas.shape}")

print(f"Magnitude of first omega: {np.linalg.norm(omegas[0]):.6f}")

Omegas shape: (99, 3)
Magnitude of first omega: 1.000000


In [ ]:
# Building a frame (gp_Ax2) at the start point using Frenet frame axes
t_arr = normalize(tangents[0])
T0    = gp_Dir(*t_arr)

# Normal from curve does not give a perfect normal to the curve, so it should be made
# perpendicular by removing the component of T0 from it.
normal_approx = to_np_vec(curve.DN(tmin, 2))
n_arr = normalize(normal_approx - np.dot(normal_approx, t_arr) * t_arr)
N0    = gp_Dir(*n_arr)

frame_origin = curve.Value(tmin)
frame        = gp_Ax2(frame_origin, T0, N0)
print(f"dot product of T0 and N0: {np.dot(t_arr, n_arr):.6e}")  # Should be ~0


dot product of T0 and N0: 1.040834e-16


In [ ]:
# Transporting the frame along the curve using the tangents and omegas

T_curr = t_arr.copy()
N_curr = n_arr.copy()

transported_frames = [(T_curr.copy(), N_curr.copy())]

for i in range(len(tangents) - 1):
    T_next = tangents[i + 1]

    cos_theta = np.clip(np.dot(T_curr, T_next), -1.0, 1.0)
    sin_theta = np.linalg.norm(np.cross(T_curr, T_next))
    theta     = np.arctan2(sin_theta, cos_theta)

    if theta < 1e-10:
        N_next = N_curr.copy()
    else:
        omega_hat = omegas[i]
        N_par  = np.dot(N_curr, omega_hat) * omega_hat
        N_perp = N_curr - N_par
        N_next = N_par + N_perp * cos_theta + np.cross(omega_hat, N_perp) * sin_theta

    N_next = N_next - np.dot(N_next, T_next) * T_next
    N_next = N_next / np.linalg.norm(N_next)

    transported_frames.append((T_next.copy(), N_next.copy()))

    T_curr = T_next
    N_curr = N_next

print(f"Transported {len(transported_frames)} frames")
T_f, N_f = transported_frames[-1]
B_f = np.cross(T_f, N_f)
print(f"T·N = {np.dot(T_f, N_f):.2e}")
print(f"T·B = {np.dot(T_f, B_f):.2e}")
print(f"N·B = {np.dot(N_f, B_f):.2e}")

Transported 100 frames
T·N = 2.78e-17
T·B = 5.55e-17
N·B = 5.55e-17


In [ ]:
# Checking the closure of the frame by comparing the first and last normals. Making correction if needed.

T_first, N_first = transported_frames[0]
T_last,  N_last  = transported_frames[-1]

sin_phi = np.dot(np.cross(N_first, N_last), T_first)
cos_phi = np.dot(N_first, N_last)
phi     = np.arctan2(sin_phi, cos_phi)   # signed, no if needed

print(f"Signed holonomy = {np.degrees(phi):.4f}°")

N_total = len(transported_frames)
corrected_frames = []

for i, (T, N) in enumerate(transported_frames):

    theta_correction = phi * i / (N_total - 1)

    cos_c = np.cos(theta_correction)
    sin_c = np.sin(theta_correction)

    B = np.cross(T, N)

    N_corrected = cos_c * N - sin_c * B
    N_corrected = N_corrected / np.linalg.norm(N_corrected)

    corrected_frames.append((T.copy(), N_corrected.copy()))

# ── Check closure ─────────────────────────────────────
T_first_c, N_first_c = corrected_frames[0]
T_last_c,  N_last_c  = corrected_frames[-1]

cos_after = np.clip(np.dot(N_first_c, N_last_c), -1.0, 1.0)
holonomy_after = np.degrees(np.arccos(cos_after))

print(f"Holonomy after  correction : {holonomy_after:.4f}°   ← ~0")

Signed holonomy = -38.5584°
Holonomy after  correction : 0.0000°   ← ~0


In [ ]:
# ── Sample positions along the curve ─────────────────
positions = []
for t in s_samples:
    p = curve.Value(t)
    positions.append(np.array([p.X(), p.Y(), p.Z()]))

positions = np.array(positions)   # shape (200, 3)
print(f"Positions shape: {positions.shape}")
print(f"First p: {positions[0]}")
print(f"Last  p: {positions[-1]}")

Positions shape: (100, 3)
First p: [ 1.4125      0.345      -1.10416667]
Last  p: [ 1.4125      0.345      -1.10416667]


In [ ]:
# Making curve into a topological object that can be swept

maker = BRepBuilderAPI_MakeEdge(curve, tmin, tmax)

if not maker.IsDone():
    raise RuntimeError("Edge creation failed")


edge = maker.Edge()

wire_maker = BRepBuilderAPI_MakeWire(edge)
if not wire_maker.IsDone():
    raise RuntimeError("Wire creation failed")

wire = wire_maker.Wire()

In [ ]:
# ── Rectangle at each frame ───────────────────────────
from OCP.BRep      import BRep_Builder
from OCP.TopoDS    import TopoDS_Compound
from OCP.BRepTools import BRepTools

w = 0.15   # width  in N direction
h = 0.25   # height in B direction

builder  = BRep_Builder()
compound = TopoDS_Compound()
builder.MakeCompound(compound)

for i, (T, N) in enumerate(corrected_frames):
    p = positions[i]
    B = np.cross(T, N)

    # 4 corners in N-B plane centered at p
    c1 = p + (w/2)*N + (h/2)*B
    c2 = p - (w/2)*N + (h/2)*B
    c3 = p - (w/2)*N - (h/2)*B
    c4 = p + (w/2)*N - (h/2)*B

    # 4 edges
    e1 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c1), gp_Pnt(*c2)).Edge()
    e2 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c2), gp_Pnt(*c3)).Edge()
    e3 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c3), gp_Pnt(*c4)).Edge()
    e4 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c4), gp_Pnt(*c1)).Edge()

    wm   = BRepBuilderAPI_MakeWire(e1, e2, e3, e4)
    face = BRepBuilderAPI_MakeFace(wm.Wire(), True).Face()

    builder.Add(compound, face)

# Add the curve wire too
builder.Add(compound, wire)

BRepTools.Write_s(compound, "rectangles_on_curve.brep")
print(f"Exported {len(corrected_frames)} rectangles: rectangles_on_curve.brep")

Exported 200 rectangles: rectangles_on_curve.brep


In [ ]:
from OCP.BRepOffsetAPI import BRepOffsetAPI_ThruSections

N_STEP = 1   # every 5th frame — reduce if too heavy

loft = BRepOffsetAPI_ThruSections(True, True)  # solid=True, ruled=True

for i in range(0, len(corrected_frames), N_STEP):
    T, N = corrected_frames[i]
    p    = positions[i]
    B    = np.cross(T, N)

    c1 = p + (w/2)*N + (h/2)*B
    c2 = p - (w/2)*N + (h/2)*B
    c3 = p - (w/2)*N - (h/2)*B
    c4 = p + (w/2)*N - (h/2)*B

    e1 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c1), gp_Pnt(*c2)).Edge()
    e2 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c2), gp_Pnt(*c3)).Edge()
    e3 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c3), gp_Pnt(*c4)).Edge()
    e4 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c4), gp_Pnt(*c1)).Edge()

    wm = BRepBuilderAPI_MakeWire(e1, e2, e3, e4)
    loft.AddWire(wm.Wire())

loft.Build()

if loft.IsDone():
    BRepTools.Write_s(loft.Shape(), "swept_coil.brep")
    print("Exported: swept_coil.brep")
else:
    print("Loft failed")

Exported: swept_coil.brep


In [ ]:
# ── Second plate — flush on top of first ──────────────
offset = h   # exact flush — bottom of plate 2 = top of plate 1

loft2 = BRepOffsetAPI_ThruSections(True, True)

for i in range(0, len(corrected_frames), N_STEP):
    T, N = corrected_frames[i]
    p    = positions[i]
    B    = np.cross(T, N)

    p_offset = p + offset * B

    c1 = p_offset + (w/2)*N + (h/2)*B
    c2 = p_offset - (w/2)*N + (h/2)*B
    c3 = p_offset - (w/2)*N - (h/2)*B
    c4 = p_offset + (w/2)*N - (h/2)*B

    e1 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c1), gp_Pnt(*c2)).Edge()
    e2 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c2), gp_Pnt(*c3)).Edge()
    e3 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c3), gp_Pnt(*c4)).Edge()
    e4 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c4), gp_Pnt(*c1)).Edge()

    wm = BRepBuilderAPI_MakeWire(e1, e2, e3, e4)
    loft2.AddWire(wm.Wire())

loft2.Build()

# ── Export both plates together ────────────────────────
builder  = BRep_Builder()
compound = TopoDS_Compound()
builder.MakeCompound(compound)
builder.Add(compound, loft.Shape())
builder.Add(compound, loft2.Shape())
builder.Add(compound, wire)

BRepTools.Write_s(compound, "stacked_plates.brep")
print("Exported: stacked_plates.brep")

builder1 = BRep_Builder()
compound1 = TopoDS_Compound()
builder1.MakeCompound(compound1)

builder1.Add(compound1, loft.Shape())
builder1.Add(compound1, loft2.Shape())

BRepTools.Write_s(compound1, "stacked_plates_no_frame.brep")

Exported: stacked_plates.brep


True

In [ ]:
builder  = BRep_Builder()
compound = TopoDS_Compound()
builder.MakeCompound(compound)

w = 0.15
h = 0.25

for i in [0, -1]:
    T, N = corrected_frames[i]
    p = positions[i]
    B = np.cross(T, N)

    c1 = p + (w/2)*N + (h/2)*B
    c2 = p - (w/2)*N + (h/2)*B
    c3 = p - (w/2)*N - (h/2)*B
    c4 = p + (w/2)*N - (h/2)*B

    e1 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c1), gp_Pnt(*c2)).Edge()
    e2 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c2), gp_Pnt(*c3)).Edge()
    e3 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c3), gp_Pnt(*c4)).Edge()
    e4 = BRepBuilderAPI_MakeEdge(gp_Pnt(*c4), gp_Pnt(*c1)).Edge()

    wm   = BRepBuilderAPI_MakeWire(e1, e2, e3, e4)
    face = BRepBuilderAPI_MakeFace(wm.Wire(), True).Face()
    builder.Add(compound, face)

builder.Add(compound, wire)

BRepTools.Write_s(compound, "first_last_rect.brep")
print("Exported: first_last_rect.brep")

Exported: first_last_rect.brep


In [ ]:
# Make a visible frame that can be exported to BREP format for visualization in CAD software

b_arr = np.cross(t_arr, n_arr)

L = 0.3

p0 = to_np_pnt(curve.Value(tmin))
p0_T = p0 + t_arr * L
p0_N = p0 + n_arr * L
p0_B = p0 + b_arr * L 

from OCP.BRepBuilderAPI import BRepBuilderAPI_MakeEdge, BRepBuilderAPI_MakeWire

maker = BRepBuilderAPI_MakeEdge(curve, tmin, tmax)

if not maker.IsDone():
    raise RuntimeError("Edge creation failed")


edge = maker.Edge()

wire_maker = BRepBuilderAPI_MakeWire(edge)
if not wire_maker.IsDone():
    raise RuntimeError("Wire creation failed")

wire = wire_maker.Wire()

edge_T = BRepBuilderAPI_MakeEdge(gp_Pnt(*p0), gp_Pnt(*p0_T)).Edge()
edge_N = BRepBuilderAPI_MakeEdge(gp_Pnt(*p0), gp_Pnt(*p0_N)).Edge()
edge_B = BRepBuilderAPI_MakeEdge(gp_Pnt(*p0), gp_Pnt(*p0_B)).Edge()

# Construct a compount to hold these axis edges and curve as wire

from OCP.BRep import BRep_Builder
from OCP.TopoDS import TopoDS_Compound

builder = BRep_Builder()
compound = TopoDS_Compound()
builder.MakeCompound(compound)

builder.Add(compound, edge_T)
builder.Add(compound, edge_N)
builder.Add(compound, edge_B)
builder.Add(compound, wire)


BRepTools.Write_s(compound, "frame_and_curve.brep")



True